# 🧠 ReadWise AI — Notebook 1: CNN–LSTM Eye-Gaze Trajectory Network

**Target Architecture**: Spatial-Temporal CNN–LSTM Deep Learning Network  
**Dataset**: ETDD70 (Eye-Tracking Dyslexia Dataset, Zenodo 13332134)  
**Target Modality**: Sequential fixation trajectories $(t, x, y, \Delta x, \Delta y)$  
**Key Guarantees**: 5-Fold `GroupKFold` on `subject_id` (Zero Subject Leakage), Automatic Checkpoint & Resume.

---
### 📌 How Checkpoint / Resume Works:
- Checkpoints are saved automatically to `/kaggle/working/checkpoints/` after every epoch.
- If the Kaggle kernel times out or crashes, re-running this notebook automatically detects existing checkpoints and **resumes training from the exact fold and epoch where it stopped!**


In [ ]:
# 1. Environment & Dependencies Setup
!pip install -q torch torchvision torchaudio scikit-learn pandas numpy matplotlib seaborn tqdm

import os
import glob
import time
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')


In [ ]:
# 2. Dataset Setup (Auto-Detects Kaggle Datasets: dataetd, Dyslexia, etdd70)
import os
import glob
import zipfile

BASE_DIR = '/kaggle/working/data/etdd70'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2.1 Locate Labels CSV (e.g. in /kaggle/input/dyslexia/ or subdirectories)
label_candidates = glob.glob('/kaggle/input/**/dyslexia_class_label.csv', recursive=True)
if label_candidates:
    LABELS_CSV = label_candidates[0]
    print(f'✓ Found Labels CSV: {LABELS_CSV}')
else:
    LABELS_CSV = os.path.join(BASE_DIR, 'dyslexia_class_label.csv')
    if not os.path.exists(LABELS_CSV):
        print('Downloading dyslexia_class_label.csv from Zenodo...')
        !curl -4 -L --connect-timeout 30 -o {LABELS_CSV} 'https://zenodo.org/records/13332134/files/dyslexia_class_label.csv?download=1'

# 2.2 Locate Fixation Files (e.g. in /kaggle/input/dataetd/data/ or subdirectories)
fixation_files = glob.glob('/kaggle/input/**/*_fixations.csv', recursive=True)
if fixation_files:
    print(f'✓ Found {len(fixation_files)} fixation sequence files in mounted Kaggle Datasets.')
else:
    print('No fixation files found in /kaggle/input. Checking /kaggle/working/ or downloading from Zenodo...')
    zip_target = os.path.join(BASE_DIR, 'data.zip')
    extracted_data = os.path.join(BASE_DIR, 'extracted', 'data')
    if not os.path.exists(extracted_data):
        if not os.path.exists(zip_target):
            print('Downloading data.zip (~127MB)...')
            !curl -4 -L --connect-timeout 30 -o {zip_target} 'https://zenodo.org/records/13332134/files/data.zip?download=1'
        print('Extracting data.zip...')
        with zipfile.ZipFile(zip_target, 'r') as zf:
            zf.extractall(os.path.join(BASE_DIR, 'extracted'))
    fixation_files = glob.glob(os.path.join(BASE_DIR, 'extracted', '**', '*_fixations.csv'), recursive=True)
    print(f'Extracted and located {len(fixation_files)} fixation files.')

print(f'Labels File: {LABELS_CSV}')
print(f'Total Fixation Files Ready: {len(fixation_files)}')


In [ ]:
# 3. Preprocess Raw Fixation Sequences (T x D representation)
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

labels_df = pd.read_csv(LABELS_CSV)
print(f'Loaded labels for {len(labels_df)} subjects.')
print(f'Processing {len(fixation_files)} fixation files...')

MAX_SEQ_LEN = 150  # Pad or truncate to 150 fixations per reading trial
FEATURE_COLS = ['duration_ms', 'fix_x', 'fix_y', 'disp_x', 'disp_y']

samples = []
for fpath in tqdm(fixation_files, desc='Parsing sequences'):
    try:
        df_fix = pd.read_csv(fpath)
        if len(df_fix) < 5:
            continue
        sid = int(df_fix.iloc[0]['sid'])
        match = labels_df[labels_df['subject_id'] == sid]
        if match.empty:
            continue
        label = int(match.iloc[0]['class_id'])
        task = str(df_fix.iloc[0]['task'])
        
        # Extract numeric trajectory
        feats = df_fix[FEATURE_COLS].copy()
        feats = feats.fillna(0.0).values
        
        # Standardize within sequence to normalize screen coordinates
        seq_mean = np.mean(feats, axis=0, keepdims=True)
        seq_std = np.std(feats, axis=0, keepdims=True) + 1e-6
        feats = (feats - seq_mean) / seq_std
        
        # Pad or truncate
        seq_len = min(len(feats), MAX_SEQ_LEN)
        padded = np.zeros((MAX_SEQ_LEN, len(FEATURE_COLS)), dtype=np.float32)
        padded[:seq_len, :] = feats[:seq_len, :]
        
        samples.append({
            'subject_id': sid,
            'task': task,
            'sequence': padded,
            'seq_len': seq_len,
            'label': label
        })
    except Exception as e:
        continue

print(f'Successfully prepared {len(samples)} sequential reading trials.')


In [ ]:
# 4. PyTorch Dataset Definition
import torch
from torch.utils.data import Dataset, DataLoader

class GazeTrajectoryDataset(Dataset):
    def __init__(self, sample_list):
        self.samples = sample_list

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        seq = torch.tensor(item['sequence'], dtype=torch.float32)
        label = torch.tensor(item['label'], dtype=torch.long)
        sid = item['subject_id']
        return seq, label, sid


In [ ]:
# 5. CNN–LSTM Spatial-Temporal Neural Network Architecture
import torch
import torch.nn as nn

class CNNLSTMGazeClassifier(nn.Module):
    """
    Spatial-Temporal Deep Learning Network as proposed in conference paper Section IV.
    - 1D-CNN: Extracts spatial fixation jump features across local receptive fields.
    - Bi-LSTM: Models temporal reading flow, regressions, and hesitation sequences.
    - Embedding Head: Produces a 32-dim dense visual representation for multimodal fusion.
    """
    def __init__(self, in_features=5, cnn_channels=32, lstm_hidden=64, embedding_dim=32, num_classes=2):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=in_features, out_channels=cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Conv1d(in_channels=cnn_channels, out_channels=cnn_channels*2, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels*2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.lstm = nn.LSTM(
            input_size=cnn_channels*2,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )
        self.embedding_head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, return_embedding=False):
        x = x.transpose(1, 2)
        conv_out = self.conv_block(x)
        conv_out = conv_out.transpose(1, 2)
        lstm_out, _ = self.lstm(conv_out)
        pooled = torch.mean(lstm_out, dim=1)
        embeddings = self.embedding_head(pooled)
        logits = self.classifier(embeddings)
        if return_embedding:
            return logits, embeddings
        return logits


In [ ]:
# 6. Checkpoint & Resume Management System
import os
import torch

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'cnn_lstm_latest.pt')
BEST_MODEL_PATH = '/kaggle/working/best_cnn_lstm_gaze_model.pt'

def save_checkpoint(fold, epoch, model, optimizer, scheduler, best_auc, history):
    state = {
        'fold': fold,
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict() if scheduler else None,
        'best_auc': best_auc,
        'history': history
    }
    torch.save(state, CHECKPOINT_PATH)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        print(f'==> Resuming from existing checkpoint at: {CHECKPOINT_PATH}')
        return torch.load(CHECKPOINT_PATH, map_location=device)
    return None


In [ ]:
# 7. 5-Fold GroupKFold Cross-Validation Training Loop
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
import torch.optim as optim

NUM_FOLDS = 5
NUM_EPOCHS = 35
BATCH_SIZE = 16
LR = 1e-3

subject_ids = np.array([s['subject_id'] for s in samples])
labels = np.array([s['label'] for s in samples])

gkf = GroupKFold(n_splits=NUM_FOLDS)
fold_results = []

checkpoint = load_checkpoint()
start_fold = checkpoint['fold'] if checkpoint else 0
start_epoch = checkpoint['epoch'] + 1 if checkpoint else 0

for fold, (train_idx, val_idx) in enumerate(gkf.split(samples, labels, groups=subject_ids)):
    if fold < start_fold:
        print(f'Skipping already completed Fold {fold+1}/{NUM_FOLDS}...')
        continue

    print(f'\n================== Fold {fold+1}/{NUM_FOLDS} ==================')
    train_samples = [samples[i] for i in train_idx]
    val_samples = [samples[i] for i in val_idx]
    
    train_loader = DataLoader(GazeTrajectoryDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(GazeTrajectoryDataset(val_samples), batch_size=BATCH_SIZE, shuffle=False)
    
    model = CNNLSTMGazeClassifier().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    
    best_val_auc = 0.0
    if checkpoint and fold == start_fold:
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        if checkpoint['scheduler_state']:
            scheduler.load_state_dict(checkpoint['scheduler_state'])
        best_val_auc = checkpoint['best_auc']
        print(f'Resumed Fold {fold+1} at Epoch {start_epoch}')
    else:
        start_epoch = 0
    
    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        total_loss = 0.0
        for seqs, targets, _ in train_loader:
            seqs, targets = seqs.to(device), targets.to(device)
            optimizer.zero_grad()
            logits = model(seqs)
            loss = criterion(logits, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
        
        # Validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for seqs, targets, _ in val_loader:
                seqs = seqs.to(device)
                probs = torch.softmax(model(seqs), dim=1)[:, 1].cpu().numpy()
                val_preds.extend(probs)
                val_targets.extend(targets.numpy())
        
        val_auc = roc_auc_score(val_targets, val_preds)
        val_acc = accuracy_score(val_targets, (np.array(val_preds) >= 0.5).astype(int))
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
        
        save_checkpoint(fold, epoch, model, optimizer, scheduler, best_val_auc, history=None)
        
        if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
            print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss: {total_loss/len(train_loader):.4f} | Val AUC: {val_auc:.3f} | Best AUC: {best_val_auc:.3f}')
            
    fold_results.append(best_val_auc)

print('\n' + '='*60)
print(f'CNN–LSTM 5-Fold GroupKFold Mean ROC-AUC: {np.mean(fold_results):.3f} +/- {np.std(fold_results):.3f}')
print('='*60)


In [ ]:
# 8. Extract & Save 32-dim Spatial-Temporal Gaze Embeddings for Notebook 3
import torch
import pandas as pd
from torch.utils.data import DataLoader

print('Generating Gaze Latent Embeddings for Multimodal Fusion...')
model = CNNLSTMGazeClassifier().to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

all_loader = DataLoader(GazeTrajectoryDataset(samples), batch_size=BATCH_SIZE, shuffle=False)
embeddings_list = []

with torch.no_grad():
    for seqs, targets, sids in all_loader:
        seqs = seqs.to(device)
        _, emb = model(seqs, return_embedding=True)
        emb_np = emb.cpu().numpy()
        for i in range(len(sids)):
            row = {'subject_id': int(sids[i]), 'label': int(targets[i])}
            for dim in range(emb_np.shape[1]):
                row[f'gaze_emb_{dim}'] = float(emb_np[i, dim])
            embeddings_list.append(row)

emb_df = pd.DataFrame(embeddings_list)
emb_csv_path = '/kaggle/working/gaze_cnn_lstm_embeddings.csv'
emb_df.to_csv(emb_csv_path, index=False)
print(f'Saved {len(emb_df)} Gaze Latent Embeddings to: {emb_csv_path}')
